SUMMARY OF PROCESSES, Reading data from previous steps and proposing steps fordward
From Aroa's notebooks I take the cleanned data and perform a new exploration to understand possible correlations and dimensions for Data Analisis


In [1]:
import pandas as pd
from project_template.paths import PROCESSED_DIR

events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
clients = pd.read_parquet(PROCESSED_DIR / "clients.parquet")

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

from project_template.paths import PROCESSED_DIR
from project_template.config import CONFIG

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)

# Vanguard's palette, so charts match the deck
MAROON, NAVY = "#96151D", "#21384E"
PALETTE = {"Control": NAVY, "Test": MAROON}

ALPHA = CONFIG["statistics"]["alpha"]
ALPHA

0.05

In [3]:
clients = pd.read_parquet(PROCESSED_DIR / "clients.parquet")

print(f"{len(clients):,} clients")
print(clients.Variation.value_counts().to_string())
clients.head()

50,488 clients
Variation
Test       26961
Control    23527


,client_id,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,9988021,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
1,8320017,Test,22.0,274.0,34.5,M,2.0,36001.90,5.0,8.0
2,4033851,Control,12.0,149.0,63.5,M,2.0,142642.26,5.0,8.0
3,1982004,Test,6.0,80.0,44.5,U,2.0,30231.76,1.0,4.0
4,9294070,Control,5.0,70.0,29.0,U,2.0,34254.54,0.0,3.0


In [4]:
NUMERIC = ["clnt_age", "clnt_tenure_yr", "bal", "num_accts",
           "calls_6_mnth", "logons_6_mnth"]

clients[NUMERIC].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
clnt_age,50487.0,47.3,15.5,17.0,33.5,48.0,59.5,96.0
clnt_tenure_yr,50488.0,12.0,6.9,2.0,6.0,11.0,16.0,55.0
bal,50488.0,149514.7,302036.4,23789.4,39878.4,65733.6,139956.5,16320040.2
num_accts,50488.0,2.3,0.5,1.0,2.0,2.0,2.0,7.0
calls_6_mnth,50488.0,3.1,2.2,0.0,1.0,3.0,5.0,6.0
logons_6_mnth,50488.0,6.1,2.2,3.0,4.0,6.0,8.0,9.0


Building the KPIs
3.0-aroa-journey-patterns.ipynb settled six reconstruction rules by reading real visits. This notebook applies them to the whole dataset and turns the event log into the measures the experiment will be judged on.

Every team computes the same KPIs. What differs is the implementation, so each one below states how it is computed, which observations it includes, and what it assumes.

The rules being applied:

A visit is one journey; its first start opens it
The first confirm closes it
The client is the unit of analysis; the visit is reported alongside
Completing means reaching confirm — for a client, in any visit
Visits with no measurable duration count in rates, not in times
Backward movement is measured, not assumed to be an error

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from project_template.paths import PROCESSED_DIR
from project_template.config import CONFIG

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 60)

MAROON, NAVY, GREY = "#96151D", "#21384E", "#857B7E"
PALETTE = {"Control": NAVY, "Test": MAROON}

FUNNEL = CONFIG["funnel"]
STEP_ORDER = {step: i for i, step in enumerate(FUNNEL)}
CONFIRM_RANK = STEP_ORDER["confirm"]

events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
clients = pd.read_parquet(PROCESSED_DIR / "clients.parquet")

events = events.sort_values(["visit_id", "date_time"]).reset_index(drop=True)
print(f"{len(events):,} events   {events.visit_id.nunique():,} visits   "
      f"{events.client_id.nunique():,} clients")

317,135 events   69,185 visits   50,488 clients


In [6]:
first_start = (
    events[events.process_step == "start"]
    .groupby("visit_id").date_time.min().rename("opened_at")
)
first_confirm = (
    events[events.process_step == "confirm"]
    .groupby("visit_id").date_time.min().rename("closed_at")
)

events = events.merge(first_start, on="visit_id", how="left")
events = events.merge(first_confirm, on="visit_id", how="left")

# Inside the attempt: at or after the opening start, at or before the first confirm
events["in_journey"] = (
    (events.opened_at.notna())
    & (events.date_time >= events.opened_at)
    & (events.date_time <= events.closed_at.fillna(events.date_time.max()))
)

print(f"events inside a journey: {events.in_journey.sum():,} "
      f"({events.in_journey.mean():.1%})")
print(f"events outside:          {(~events.in_journey).sum():,}")

events inside a journey: 300,982 (94.9%)
events outside:          16,153


Journey-level measures

In [7]:
inside = events[events.in_journey].copy()

# Time between consecutive events, and whether the step advanced
inside["gap_s"] = inside.groupby("visit_id").date_time.diff().dt.total_seconds()
inside["advanced"] = inside.step_rank.diff() > 0
inside.loc[inside.groupby("visit_id").cumcount() == 0, "advanced"] = False

by_visit = inside.groupby("visit_id")

journeys = pd.DataFrame({
    "client_id": by_visit.client_id.first(),
    "group": by_visit.Variation.first(),
    "events": by_visit.size(),
    "max_step": by_visit.step_rank.max(),
    "opened_at": by_visit.date_time.min(),
    "step_backs": by_visit.is_backward.sum(),
    "repeated_steps": by_visit.is_repeat.sum(),
})

journeys["completed"] = journeys.max_step == CONFIRM_RANK

# Rule 5: durations only where there is something to measure
duration = (by_visit.date_time.max() - by_visit.date_time.min()).dt.total_seconds()
journeys["completion_time_s"] = np.where(
    journeys.completed & (duration > 0), duration, np.nan
)

# Effective time: only the intervals in which the client moved forward
forward = inside[inside.advanced].groupby("visit_id").gap_s.sum()
journeys["effective_time_s"] = journeys.index.map(forward).where(journeys.completed)

print(f"{len(journeys):,} journeys   {journeys.completed.sum():,} completed "
      f"({journeys.completed.mean():.1%})")
journeys.head()

63,885 journeys   32,658 completed (51.1%)


,client_id,group,events,max_step,opened_at,step_backs,repeated_steps,completed,completion_time_s,effective_time_s
visit_id,,,,,,,,,,
100019538_17884295066_43909,7338123,Test,11,4,2017-04-09 16:20:56,2,2,True,242.0,204.0
100022086_87870757897_149620,2478628,Test,5,4,2017-05-23 20:44:01,0,0,True,180.0,180.0
100030127_47967100085_936361,105007,Control,1,0,2017-03-22 11:07:49,0,0,False,NaN,NaN
100037962_47432393712_705583,5623007,Control,4,1,2017-04-14 16:41:51,1,1,False,NaN,NaN
100057941_88477660212_944512,4823947,Control,7,3,2017-04-09 11:30:10,1,0,False,NaN,NaN
